# Forecasting Pipeline

DACON 온라인 채널 제품 판매량 예측 파이프라인입니다. 제품별 시계열 학습 데이터를 만들고, 캘린더/lag/rolling 피처를 생성한 뒤 LightGBM 회귀로 미래 판매량을 재귀적으로 예측합니다.


## Imports & Setup

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from lightgbm import LGBMRegressor
except ImportError:
    from sklearn.ensemble import HistGradientBoostingRegressor as LGBMRegressor

In [ ]:
SEED = 42
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)

DATA_DIR = Path("data")
OUTPUT_PATH = Path("submissions/sales_forecast_submission.csv")
TRAIN_FILE = "train.csv"
SAMPLE_SUBMISSION_FILE = "sample_submission.csv"

ITEM_ID_COLUMN = "ID"
TIMESTAMP_COLUMN = "date"
TARGET_COLUMN = "sales"
FORECAST_COLUMN = "prediction"

LAG_DAYS = (1, 7, 14, 28)
ROLLING_WINDOWS = (7, 14, 28)
MIN_TRAINING_LAG_DAYS = 28

## Load Data

In [ ]:
train = pd.read_csv(DATA_DIR / TRAIN_FILE)
sample_submission = pd.read_csv(DATA_DIR / SAMPLE_SUBMISSION_FILE)
print(f"train: {train.shape}, sample_submission: {sample_submission.shape}")
train.head()

## Wide → Long 변환

판매량 wide 테이블 (제품 × 날짜) 을 long 형식으로 변환합니다.


In [ ]:
def is_date_like(value):
    try:
        pd.to_datetime(str(value), errors="raise")
        return True
    except (ValueError, TypeError):
        return False


def infer_date_columns(frame):
    date_columns = [c for c in frame.columns if is_date_like(c)]
    if not date_columns:
        raise ValueError("No date-like columns were found.")
    return date_columns


def infer_item_id_column(frame, preferred):
    return preferred if preferred in frame.columns else frame.columns[0]

In [ ]:
date_columns = infer_date_columns(train)
item_id_column = infer_item_id_column(train, ITEM_ID_COLUMN)
metadata_columns = [c for c in train.columns if c not in {item_id_column, *date_columns}]

train_long = train.melt(
    id_vars=[item_id_column, *metadata_columns],
    value_vars=date_columns,
    var_name=TIMESTAMP_COLUMN,
    value_name=TARGET_COLUMN,
)
train_long = train_long.rename(columns={item_id_column: ITEM_ID_COLUMN})
train_long[TIMESTAMP_COLUMN] = pd.to_datetime(train_long[TIMESTAMP_COLUMN])
train_long[TARGET_COLUMN] = pd.to_numeric(train_long[TARGET_COLUMN], errors="coerce").fillna(0).clip(lower=0)
train_long = train_long.sort_values([ITEM_ID_COLUMN, TIMESTAMP_COLUMN]).reset_index(drop=True)
train_long.head()

## Sample Submission → Long

미래 예측이 필요한 (제품, 날짜) 조합을 long 형식으로 만들고 메타데이터를 join 합니다.


In [ ]:
submission_date_columns = infer_date_columns(sample_submission)
submission_id_column = infer_item_id_column(sample_submission, ITEM_ID_COLUMN)

metadata = train[[item_id_column, *metadata_columns]].rename(columns={item_id_column: ITEM_ID_COLUMN}).drop_duplicates(subset=[ITEM_ID_COLUMN])

future_long = sample_submission.melt(
    id_vars=[submission_id_column],
    value_vars=submission_date_columns,
    var_name="date_column",
    value_name=FORECAST_COLUMN,
)
future_long = future_long.rename(columns={submission_id_column: ITEM_ID_COLUMN})
future_long[TIMESTAMP_COLUMN] = pd.to_datetime(future_long["date_column"])
future_long = future_long.merge(metadata, on=ITEM_ID_COLUMN, how="left")
future_long = future_long.sort_values([TIMESTAMP_COLUMN, ITEM_ID_COLUMN]).reset_index(drop=True)
future_long.head()

## Feature Engineering

In [ ]:
def add_calendar_features(frame):
    enriched = frame.copy()
    ts = enriched[TIMESTAMP_COLUMN]
    enriched["year"] = ts.dt.year
    enriched["month"] = ts.dt.month
    enriched["day"] = ts.dt.day
    enriched["day_of_week"] = ts.dt.dayofweek
    enriched["week_of_year"] = ts.dt.isocalendar().week.astype(int)
    enriched["is_weekend"] = enriched["day_of_week"].isin([5, 6]).astype(int)
    enriched["month_sin"] = np.sin(2 * np.pi * enriched["month"] / 12)
    enriched["month_cos"] = np.cos(2 * np.pi * enriched["month"] / 12)
    enriched["dow_sin"] = np.sin(2 * np.pi * enriched["day_of_week"] / 7)
    enriched["dow_cos"] = np.cos(2 * np.pi * enriched["day_of_week"] / 7)
    return enriched


def add_lag_features(frame):
    featured = frame.sort_values([ITEM_ID_COLUMN, TIMESTAMP_COLUMN]).copy()
    grouped = featured.groupby(ITEM_ID_COLUMN)[TARGET_COLUMN]
    for lag in LAG_DAYS:
        featured[f"lag_{lag}"] = grouped.shift(lag)
    shifted = grouped.shift(1)
    for window in ROLLING_WINDOWS:
        rolling = shifted.groupby(featured[ITEM_ID_COLUMN]).rolling(window=window, min_periods=1)
        featured[f"rolling_mean_{window}"] = rolling.mean().reset_index(level=0, drop=True)
        featured[f"rolling_std_{window}"] = rolling.std().reset_index(level=0, drop=True).fillna(0)
    return featured


def get_model_feature_columns(frame):
    reserved = {TIMESTAMP_COLUMN, TARGET_COLUMN, FORECAST_COLUMN, "date_column"}
    return [c for c in frame.columns if c not in reserved]

In [ ]:
training_frame = add_lag_features(add_calendar_features(train_long))
required_lag = f"lag_{MIN_TRAINING_LAG_DAYS}"
if required_lag in training_frame.columns:
    training_frame = training_frame.dropna(subset=[required_lag])
training_frame = training_frame.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)

feature_columns = get_model_feature_columns(training_frame)
train_features = training_frame[feature_columns]
train_target = training_frame[TARGET_COLUMN]
print(f"training rows: {len(training_frame)}, features: {len(feature_columns)}")

## Modeling

LightGBM 회귀 + log1p 타겟 변환. numeric/categorical 전처리는 sklearn `ColumnTransformer` 로 묶었습니다.


In [ ]:
def build_regressor(seed=SEED):
    return LGBMRegressor(
        n_estimators=600,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=seed,
        objective="regression_l1",
        verbose=-1,
    )


def build_model(features):
    numeric_columns = features.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_columns = [c for c in features.columns if c not in numeric_columns]

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", SimpleImputer(strategy="median"), numeric_columns),
            (
                "categorical",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]),
                categorical_columns,
            ),
        ],
        remainder="drop",
    )

    regressor = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", build_regressor()),
    ])

    return TransformedTargetRegressor(
        regressor=regressor,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )

In [ ]:
model = build_model(train_features)
model.fit(train_features, train_target.clip(lower=0))

## Recursive Forecasting

미래 시점을 한 단계씩 예측하면서, 직전 예측값을 history 에 누적시켜 다음 시점의 lag/rolling 피처를 만듭니다.


In [ ]:
history = train_long.copy()
predictions = []

for forecast_date in sorted(future_long[TIMESTAMP_COLUMN].unique()):
    future_step = future_long[future_long[TIMESTAMP_COLUMN] == forecast_date].copy()
    future_step[TARGET_COLUMN] = np.nan

    combined = pd.concat([history, future_step], ignore_index=True, sort=False)
    featured = add_lag_features(add_calendar_features(combined))
    future_features_frame = featured.loc[featured[TARGET_COLUMN].isna()].copy()
    future_features = future_features_frame[get_model_feature_columns(future_features_frame)]

    forecast = np.maximum(model.predict(future_features), 0)
    predicted_step = future_step.copy()
    predicted_step[FORECAST_COLUMN] = forecast
    predictions.append(predicted_step[[ITEM_ID_COLUMN, "date_column", FORECAST_COLUMN]])

    history_step = future_step.copy()
    history_step[TARGET_COLUMN] = forecast
    history = pd.concat([history, history_step], ignore_index=True, sort=False)

prediction_frame = pd.concat(predictions, ignore_index=True)
prediction_frame.head()

## Build Submission

In [ ]:
submission = sample_submission.copy()
prediction_lookup = prediction_frame.set_index([ITEM_ID_COLUMN, "date_column"])[FORECAST_COLUMN]
submission_date_columns = infer_date_columns(submission)

for date_column in submission_date_columns:
    keys = list(zip(submission[submission_id_column], [date_column] * len(submission)))
    submission[date_column] = [prediction_lookup.get(key, 0.0) for key in keys]

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)
print(f"saved: {OUTPUT_PATH}")
submission.head()